# 3. 에이전트 오케스트레이션

**exercise 3 | 에이전트 유형별 오케스트레이션 × APD 실습**

---


## 학습 목표
1. **반사형 / ReAct / 계획형 / 성찰형** 에이전트 유형별 오케스트레이션 직접 구현
2. **exercise 1, 2에서 만든 에이전트**를 클래스 인터페이스로 재사용
3. APD **전체 파이프라인**을 멀티 에이전트 시스템으로 실행
4. 순차/병렬 패턴 비교 — `ThreadPoolExecutor`로 Task 1 병렬 심사

## 이번 exercise 위치: APD 파이프라인 전체

```
INPUT(지원서) → [Task 0: Application Collector 🟦]
 │
 ┌──────────┼──────────┐
 ▼ ▼ ▼
 [Task 1a: Essay [Task 1b: Elig. [Task 1c: Portfolio
 Analyzer 🟧] Checker 🟧] Evaluator 🟧]
 └──────────┴──────────┘
 │
 ▼
 [Task 2: Selection Recommender 🟪]
 │
 ▼
 [Task 3: 면접 (Human) 🟩]
 │
 ▼
 [Task 4: Acceptance Recommender 🟪]
 │
 ▼
 [Task 5: Notification Sender 🟩]
 │
 ▼
 OUTPUT(합격자)
```

| exercise 3 구현 | APD Task | 에이전트 유형 |
|------------|----------|------------|
| Application Router | Task 0 보조 | 반사형 (Reflex) |
| Selection Recommender | Task 2 | ReAct |
| Acceptance Recommender | Task 4 | 계획형 + 쿼리 분해 |
| Notification Sender | Task 5 | 성찰형 (Reflection) |
| 파이프라인 오케스트레이터 | 전체 | 멀티 에이전트 |

**exercise 1, 2에서 구현한 에이전트** (재사용):
- `EligibilityCheckerAgent` — Task 1b (exercise 1)
- `ApplicationCollectorAgent` — Task 0 (exercise 2)
- `EssayAnalyzerAgent` — Task 1a (exercise 2)
- `PortfolioEvaluatorAgent` — Task 1c (exercise 2)

In [1]:
# 필요한 라이브러리 설치
!pip install -q google-genai requests

from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.colab import auth
import json
import os
import re
import asyncio
import time
import requests
from datetime import datetime

# Vertex AI 인증 (Google 계정으로 로그인)
auth.authenticate_user()

PROJECT_ID = "scientific-management-494205"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1"),
)
print('✅ 환경 설정 완료')

✅ 환경 설정 완료


## 1. 에이전트 유형별 오케스트레이션: 복잡도 스펙트럼

에이전트 유형에 따라 추론, 계획, 행동의 방식이 달라집니다.
작업 복잡도에 맞는 유형을 선택하는 것이 핵심입니다.

```
복잡도 낮음 ─────────────────────────────── 복잡도 높음

 반사형 ReAct 계획형 성찰형
(즉각 응답) (추론+행동) (계획→실행) (자기평가+수정)
 ↓ ↓ ↓ ↓
Application Selection Acceptance Notification
 Router Recommender Recommender Sender
 (Task 0) (Task 2) (Task 4) (Task 5)
```

| 유형 | 핵심 원리 | APD 적용 에이전트 |
|------|---------|----------------|
| **반사형** | if-condition → action, LLM 없음 | Application Router |
| **ReAct** | Thought → Action → Observation 루프 | Selection Recommender |
| **계획형 + 쿼리 분해** | 계획 수립 후 실행 분리 | Acceptance Recommender |
| **성찰형** | 생성 → 자기 평가 → 재생성 | Notification Sender |

In [2]:
# ====================================================
# 셋업: exercise 1, 2 에이전트 클래스 재정의
# (Colab은 노트북 간 상태 공유 불가 → 간결하게 재정의)
# ====================================================

class BaseAgentWorker:
    """APD 에이전트 워커의 공통 인터페이스.
    모든 에이전트는 run() 메서드를 구현해야 합니다.
    오케스트레이터는 에이전트 유형에 관계없이 run()으로 호출합니다.
    """
    def __init__(self, name: str, task_id: str):
        self.name = name
        self.task_id = task_id
    def run(self, input_text: str) -> str:
        raise NotImplementedError(f'{self.__class__.__name__}.run()을 구현하세요.')
    def __repr__(self):
        return f'{self.__class__.__name__}(name="{self.name}", task="{self.task_id}")'


# ====================================================
# exercise 1: Eligibility Checker (Task 1b)
# ====================================================
ELIGIBILITY_SYSTEM_PROMPT = '''당신은 대학교 AI 동아리의 신입 부원 자격을 검사하는 Eligibility Checker 에이전트입니다.
[자격 기준]
1. 학년: 1~3학년만 지원 가능 (4학년 지원 불가)
2. 전공: 공과대학 또는 자연과학대학 소속 (컴퓨터공학과, 전기공학과, 물리학과, 수학과, 통계학과 등)
3. 이수 과목: 파이썬 기초 또는 선형대수 중 최소 1개
[응답 형식]
- 학년 기준: ✅ 합격 또는 ❌ 불합격 (이유)
- 전공 기준: ✅ 합격 또는 ❌ 불합격 (이유)
- 이수 과목: ✅ 합격 또는 ❌ 불합격 (이유)
- 최종 판정: **PASS** 또는 **FAIL** — (사유)'''

class EligibilityCheckerAgent(BaseAgentWorker):
    """Task 1b — Eligibility Checker (exercise 1에서 구현)."""
    def __init__(self):
        super().__init__('Eligibility Checker', 'Task 1b')
        self.config = types.GenerateContentConfig(
            system_instruction=ELIGIBILITY_SYSTEM_PROMPT,
            thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
    def run(self, applicant_info: str) -> str:
        return client.models.generate_content(
            model='gemini-2.5-flash', contents=applicant_info, config=self.config).text


# ====================================================
# exercise 2: 로컬 도구 함수
# ====================================================
processed_ids: set = set()
MAJOR_ALIASES = {
    '컴공': '컴퓨터공학과', '컴퓨터공학': '컴퓨터공학과',
    '전기전자': '전기공학과', '물리': '물리학과',
    '수학': '수학과', '통계': '통계학과',
    '소웨': '소프트웨어학과', '소프트웨어': '소프트웨어학과',
}

def validate_fields(name: str, student_id: str, major: str, courses: str) -> str:
    '''지원서 필수 필드의 존재 여부와 형식을 검사합니다.'''
    errors = []
    if not name or not name.strip():
        errors.append('이름이 비어있습니다')
    if not student_id:
        errors.append('학번이 비어있습니다')
    elif not re.fullmatch(r'\d{8}', student_id.strip()):
        errors.append(f'학번 형식 오류: "{student_id}" (8자리 숫자 필요)')
    if not major or not major.strip():
        errors.append('전공이 비어있습니다')
    if errors:
        return '❌ 유효성 오류: ' + ' / '.join(errors)
    return f'✅ VALID: {name} ({student_id}, {major})'

def check_duplicate(student_id: str) -> str:
    '''이미 처리된 학번인지 확인하여 중복 지원을 방지합니다.'''
    sid = student_id.strip()
    if sid in processed_ids:
        return f'⚠️  DUPLICATE: 학번 {sid}는 이미 처리된 지원서입니다.'
    return f'✅ NEW: 학번 {sid}는 신규 지원자입니다.'

def normalize_application(name: str, student_id: str, major: str, courses: str) -> str:
    '''전공명 표준화 및 과목 목록 정제 후 정규화된 지원서를 반환합니다.'''
    normalized_major = MAJOR_ALIASES.get(major.strip(), major.strip())
    course_list = list(dict.fromkeys([c.strip() for c in courses.split(',') if c.strip()]))
    processed_ids.add(student_id.strip())
    result = {'name': name.strip(), 'student_id': student_id.strip(),
              'major': normalized_major, 'courses': course_list,
              'processed_at': datetime.now().strftime('%Y-%m-%d %H:%M')}
    return f'✅ 정규화 완료: {json.dumps(result, ensure_ascii=False)}'

def search_wikipedia(query: str, max_chars: int = 500) -> str:
    '''Wikipedia에서 기술 개념이나 용어를 검색합니다.'''
    try:
        encoded = requests.utils.quote(query)
        headers = {'User-Agent': 'LET-Project/1.0 (educational; python-requests)'}
        resp = requests.get(f'https://ko.wikipedia.org/api/rest_v1/page/summary/{encoded}',
                            timeout=5, headers=headers)
        if resp.status_code == 200:
            extract = resp.json().get('extract', '')
            return extract[:max_chars] + ('...' if len(extract) > max_chars else '')
        resp_en = requests.get(f'https://en.wikipedia.org/api/rest_v1/page/summary/{encoded}',
                               timeout=5, headers=headers)
        if resp_en.status_code == 200:
            extract = resp_en.json().get('extract', '')
            return extract[:max_chars] + ('...' if len(extract) > max_chars else '')
        return f'\'{query}\' 관련 정보를 찾을 수 없습니다.'
    except Exception as e:
        return f'검색 오류: {str(e)}'

def analyze_portfolio_image(image_url: str, evaluation_rubric: str = '') -> str:
    '''지원자의 포트폴리오 이미지를 Gemini Vision으로 분석합니다.
    이미지 다운로드 실패 시 URL 정보 기반 텍스트 평가로 폴백합니다.
    '''
    rubric_text = evaluation_rubric or '작업물의 품질, 완성도, AI/기술 관련성'
    # 1차 시도: 이미지 바이트 다운로드 후 Vision API 호출
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (compatible; LET-Project/1.0)'}
        resp = requests.get(image_url, timeout=10, headers=headers)
        resp.raise_for_status()
        mime_type = resp.headers.get('Content-Type', 'image/jpeg').split(';')[0].strip()
        if not mime_type.startswith('image/'):
            mime_type = 'image/jpeg'
        image_part = types.Part.from_bytes(data=resp.content, mime_type=mime_type)
        q = (f'이 포트폴리오 이미지를 동아리 지원자 작업물로서 평가해주세요.\n'
             f'평가 기준: {rubric_text}\n각 기준을 1~5점으로 평가하고 종합 의견을 제시해주세요.')
        return client.models.generate_content(model='gemini-2.5-flash', contents=[q, image_part]).text
    except Exception:
        pass  # 네트워크 오류 시 텍스트 폴백으로 진행

    # 폴백: 이미지 URL 기반 텍스트 평가
    try:
        fallback_q = (
            f'포트폴리오 URL: {image_url}\n\n'
            f'이미지를 직접 열 수 없지만, URL 경로와 파일명 정보를 바탕으로 '
            f'동아리 지원자 포트폴리오를 가정적으로 평가해주세요.\n'
            f'평가 기준: {rubric_text}\n'
            f'각 기준을 1~5점으로 평가하고 [네트워크 폴백 평가]임을 명시해주세요.'
        )
        return client.models.generate_content(model='gemini-2.5-flash', contents=fallback_q).text
    except Exception as e:
        return f'포트폴리오 분석 불가: {str(e)}'


# ====================================================
# exercise 2: 에이전트 클래스
# ====================================================
class ApplicationCollectorAgent(BaseAgentWorker):
    """Task 0 — Application Collector (exercise 2에서 구현)."""
    def __init__(self):
        super().__init__('Application Collector', 'Task 0')
        self._chat = client.chats.create(
            model='gemini-2.5-flash',
            config=types.GenerateContentConfig(
                tools=[validate_fields, check_duplicate, normalize_application],
                system_instruction='동아리 지원서 수집·검증 에이전트. check_duplicate → validate_fields → normalize_application 순으로 처리하세요.',
                thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
        )
    def run(self, application_text: str) -> str:
        return self._chat.send_message(application_text).text

class EssayAnalyzerAgent(BaseAgentWorker):
    """Task 1a — Essay Analyzer (exercise 2에서 구현)."""
    def __init__(self):
        super().__init__('Essay Analyzer', 'Task 1a')
        self._essay_config = types.GenerateContentConfig(
            tools=[search_wikipedia],
            system_instruction='에세이 심사 에이전트. 기술 용어를 Wikipedia로 검색하여 기술 이해도/동아리 적합성/글쓰기 명확성을 각 1~5점으로 평가하세요.',
            thinking_config=types.ThinkingConfig(thinking_budget=0)
        )
    def run(self, essay_text: str) -> str:
        chat = client.chats.create(model='gemini-2.5-flash', config=self._essay_config)
        return chat.send_message(f'에세이 심사:\n{essay_text}').text

class PortfolioEvaluatorAgent(BaseAgentWorker):
    """Task 1c — Portfolio Evaluator (exercise 2에서 구현)."""
    def __init__(self):
        super().__init__('Portfolio Evaluator', 'Task 1c')
        self._portfolio_config = types.GenerateContentConfig(
            tools=[analyze_portfolio_image],
            system_instruction='포트폴리오 평가 에이전트. 이미지 URL로 analyze_portfolio_image()를 호출하여 기술 수준/AI관련성/창의성을 각 1~5점으로 평가하세요.',
            thinking_config=types.ThinkingConfig(thinking_budget=0)
        )
    def run(self, image_url: str) -> str:
        chat = client.chats.create(model='gemini-2.5-flash', config=self._portfolio_config)
        return chat.send_message(f'포트폴리오 평가: {image_url}').text


# ====================================================
# 에이전트 목록 확인
# ====================================================
print('📋 exercise 1, 2 에이전트 로드 완료:')
all_w1w2_agents = [
    ApplicationCollectorAgent(),
    EligibilityCheckerAgent(),
    EssayAnalyzerAgent(),
    PortfolioEvaluatorAgent(),
]
for a in all_w1w2_agents:
    print(f'  ✅ {a}')
print('\n→ 이 에이전트들을 이번 파이프라인에서 재사용합니다.')

📋 exercise 1, 2 에이전트 로드 완료:
  ✅ ApplicationCollectorAgent(name="Application Collector", task="Task 0")
  ✅ EligibilityCheckerAgent(name="Eligibility Checker", task="Task 1b")
  ✅ EssayAnalyzerAgent(name="Essay Analyzer", task="Task 1a")
  ✅ PortfolioEvaluatorAgent(name="Portfolio Evaluator", task="Task 1c")

→ 이 에이전트들을 이번 주 파이프라인에서 재사용합니다.


## 2. 반사형 에이전트 (Reflex Agent) — Application Router

### APD 적용: Task 0 앞단 초기 라우터

지원서가 접수되면 **LLM 추론 없이** 즉각 유형을 분류하고 처리 경로를 결정합니다.

| 입력 조건 | 분류 | 처리 경로 |
|----------|------|----------|
| 학번이 8자리 숫자가 아닌 경우 | `ERROR` | 즉각 반려 |
| 전공에 'international', '해외' 등 포함 | `OVERSEAS` | 별도 검토 큐 |
| 전공 정보 누락 | `INCOMPLETE` | 보완 요청 |
| 그 외 정상 지원서 | `STANDARD` | Application Collector로 전달 |

> **핵심**: LLM 없음 → 밀리초 응답, 예측 가능한 성능
> **한계**: 규칙 범위 밖의 입력은 처리 불가

In [3]:
# ====================================================
# 반사형 에이전트: LLM 없음, if-else 규칙만으로 즉각 분기
# ====================================================

def classify_application(student_id: str, major: str) -> str:
    """지원서 유형을 키워드 규칙으로 즉각 분류합니다. (LLM 호출 없음)

    Args:
        student_id: 학번 문자열
        major: 전공명 문자열

    Returns:
        분류 결과: 'ERROR' | 'OVERSEAS' | 'INCOMPLETE' | 'STANDARD'
    """
    if not student_id or not re.fullmatch(r'\d{8}', student_id.strip()):
        return 'ERROR'
    overseas_keywords = ['해외', 'foreign', 'international', 'overseas', 'abroad']
    if any(k in major.lower() for k in overseas_keywords):
        return 'OVERSEAS'
    if not major.strip():
        return 'INCOMPLETE'
    return 'STANDARD'


def reflex_application_router(student_id: str, major: str, courses: str) -> dict:
    """반사형 에이전트: 규칙 기반 즉각 라우팅.

    Args:
        student_id: 지원자 학번
        major: 전공명
        courses: 이수 과목

    Returns:
        {'category': ..., 'next_agent': ..., 'message': ...} 딕셔너리
    """
    category = classify_application(student_id, major)
    routes = {
        'ERROR':      {'category': 'ERROR',
                       'next_agent': None,
                       'message': f'[반사형] ❌ 즉각 반려: 학번 형식 오류 ({student_id})'},
        'OVERSEAS':   {'category': 'OVERSEAS',
                       'next_agent': 'SPECIAL_QUEUE',
                       'message': '[반사형] ⚠️ 해외 대학 지원자 → 별도 검토 큐 전송'},
        'INCOMPLETE': {'category': 'INCOMPLETE',
                       'next_agent': None,
                       'message': '[반사형] ⚠️ 전공 정보 누락 → 지원자에게 보완 요청'},
        'STANDARD':   {'category': 'STANDARD',
                       'next_agent': 'ApplicationCollectorAgent',
                       'message': '[반사형] ✅ 정상 접수 → Application Collector로 전달'},
    }
    return routes[category]


# ====================================================
# 반사형 에이전트 테스트
# ====================================================
import time as _time

print('=' * 57)
print('🤖 반사형 에이전트 (Reflex Agent) — Application Router')
print('=' * 57)
print('특징: LLM 없음, 키워드 규칙만, 즉각 응답\n')

test_applications = [
    ('2024ABC',  '컴퓨터공학과', '파이썬 기초'),      # 학번 오류
    ('20240101', 'International Studies', 'Python'),   # 해외 대학
    ('20240202', '', '선형대수'),                      # 전공 누락
    ('20240303', '물리학과', '선형대수'),               # 정상
    ('20240404', '컴공', '파이썬 기초, 선형대수'),      # 정상 (약칭)
]

for sid, major, courses in test_applications:
    t0 = _time.time()
    result = reflex_application_router(sid, major, courses)
    elapsed_ms = (_time.time() - t0) * 1000

    print(f'👤 입력: 학번={sid} | 전공={major or "(없음)"}')
    print(f'🔍 분류: {result["category"]}')
    print(f'⚡ 응답: {result["message"]}')
    print(f'   → 다음 에이전트: {result["next_agent"]} | 응답 시간: {elapsed_ms:.2f}ms\n')

print('✅ 반사형 에이전트 완료')
print('   → 강점: 밀리초 단위 응답, 예측 가능한 성능')
print('   → 한계: 규칙 범위 밖 입력 처리 불가')
print('   → 다음: ReAct로 다단계 추론 추가')

🤖 반사형 에이전트 (Reflex Agent) — Application Router
특징: LLM 없음, 키워드 규칙만, 즉각 응답

👤 입력: 학번=2024ABC | 전공=컴퓨터공학과
🔍 분류: ERROR
⚡ 응답: [반사형] ❌ 즉각 반려: 학번 형식 오류 (2024ABC)
   → 다음 에이전트: None | 응답 시간: 0.13ms

👤 입력: 학번=20240101 | 전공=International Studies
🔍 분류: OVERSEAS
⚡ 응답: [반사형] ⚠️ 해외 대학 지원자 → 별도 검토 큐 전송
   → 다음 에이전트: SPECIAL_QUEUE | 응답 시간: 0.03ms

👤 입력: 학번=20240202 | 전공=(없음)
🔍 분류: INCOMPLETE
⚡ 응답: [반사형] ⚠️ 전공 정보 누락 → 지원자에게 보완 요청
   → 다음 에이전트: None | 응답 시간: 0.01ms

👤 입력: 학번=20240303 | 전공=물리학과
🔍 분류: STANDARD
⚡ 응답: [반사형] ✅ 정상 접수 → Application Collector로 전달
   → 다음 에이전트: ApplicationCollectorAgent | 응답 시간: 0.01ms

👤 입력: 학번=20240404 | 전공=컴공
🔍 분류: STANDARD
⚡ 응답: [반사형] ✅ 정상 접수 → Application Collector로 전달
   → 다음 에이전트: ApplicationCollectorAgent | 응답 시간: 0.00ms

✅ 반사형 에이전트 완료
   → 강점: 밀리초 단위 응답, 예측 가능한 성능
   → 한계: 규칙 범위 밖 입력 처리 불가
   → 다음: ReAct로 다단계 추론 추가


## 3. 리액트 에이전트 (ReAct Agent) — Selection Recommender (Task 2)

### APD 적용: Task 1 심사 결과 통합 → 면접 후보 선발

1차 심사 결과를 받아 Selection Recommender가
**Thought → Action → Observation** 루프로 단계적으로 후보를 분석하고 선발합니다.

```
Thought: "이지은 지원자의 점수를 먼저 조회해야겠다"
Action: get_screening_result("20240101")
Observation: essay=4.2, eligibility=PASS, portfolio=3.8

Thought: "가중치를 적용한 종합 점수를 계산해보자"
Action: calculate_weighted_score("20240101")
Observation: 종합 점수 = 3.94

...

Thought: "모든 지원자를 분석했다. 상위 후보를 추려보자"
Action: get_shortlist(min_score=3.5, max_candidates=3)
Observation: [이지은, 최수진, 박민준]

Final Answer: 면접 후보 3명 선정 완료
```

> **강점**: 탐색적 시나리오에 강함, 실행 과정 로그 기반 감사(audit) 가능
> **약점**: 루프 구조로 인한 API 비용·응답 시간 증가

In [4]:
# ====================================================
# Selection Recommender 도구 정의
# ====================================================

# 가상 1차 심사 결과 DB (Task 1a + 1b + 1c 결과 통합)
SCREENING_RESULTS_DB = {
    '20240101': {'name': '이지은',  'essay_score': 4.2, 'eligibility': 'PASS', 'portfolio_score': 3.8},
    '20240202': {'name': '박민준',  'essay_score': 3.5, 'eligibility': 'PASS', 'portfolio_score': 4.5},
    '20240303': {'name': '김철수',  'essay_score': 2.1, 'eligibility': 'FAIL', 'portfolio_score': 2.0},
    '20240404': {'name': '최수진',  'essay_score': 4.8, 'eligibility': 'PASS', 'portfolio_score': 4.2},
    '20240505': {'name': '정다은',  'essay_score': 3.9, 'eligibility': 'PASS', 'portfolio_score': 3.6},
}


def get_screening_result(student_id: str) -> str:
    """지원자의 1차 심사(에세이/자격/포트폴리오) 결과를 조회합니다.

    Args:
        student_id: 조회할 지원자 학번

    Returns:
        심사 결과 JSON 문자열. 없으면 오류 메시지.
    """
    result = SCREENING_RESULTS_DB.get(student_id.strip())
    if not result:
        return f'학번 {student_id} 심사 결과 없음'
    return json.dumps({'student_id': student_id, **result}, ensure_ascii=False)


def calculate_weighted_score(student_id: str, essay_weight: float = 0.3,
                             portfolio_weight: float = 0.7) -> str:
    """가중치를 적용한 종합 점수를 계산합니다.

    Args:
        student_id: 지원자 학번
        essay_weight: 에세이 점수 가중치 (기본 0.3)
        portfolio_weight: 포트폴리오 점수 가중치 (기본 0.7)

    Returns:
        종합 점수 계산 결과. eligibility FAIL이면 0점 처리.
    """
    result = SCREENING_RESULTS_DB.get(student_id.strip())
    if not result:
        return f'학번 {student_id} 데이터 없음'
    if result['eligibility'] == 'FAIL':
        return f'{result["name"]} ({student_id}): 종합 점수 = 0.00 (자격 미달 — eligibility FAIL)'
    score = result['essay_score'] * essay_weight + result['portfolio_score'] * portfolio_weight
    return (f'{result["name"]} ({student_id}): 종합 점수 = {score:.2f} '
            f'(에세이 {result["essay_score"]}×{essay_weight} + '
            f'포트폴리오 {result["portfolio_score"]}×{portfolio_weight})')


def get_shortlist(min_score: float = 3.5, max_candidates: int = 3) -> str:
    """점수 기준으로 면접 후보 목록을 반환합니다.

    Args:
        min_score: 최소 종합 점수 (기본 3.5)
        max_candidates: 최대 후보 수 (기본 3)

    Returns:
        후보 목록 JSON 문자열
    """
    # LLM이 float으로 전달할 수 있으므로 int 변환
    max_candidates = int(max_candidates)
    scored = []
    for sid, data in SCREENING_RESULTS_DB.items():
        if data['eligibility'] == 'FAIL':
            continue
        score = data['essay_score'] * 0.3 + data['portfolio_score'] * 0.7
        if score >= min_score:
            scored.append({'student_id': sid, 'name': data['name'], 'score': round(score, 2)})
    scored.sort(key=lambda x: x['score'], reverse=True)
    shortlist = scored[:max_candidates]
    return json.dumps({'shortlist': shortlist, 'count': len(shortlist)}, ensure_ascii=False)


TOOL_REGISTRY_SELECTION = {
    'get_screening_result': get_screening_result,
    'calculate_weighted_score': calculate_weighted_score,
    'get_shortlist': get_shortlist,
}

print('✅ Selection Recommender 도구 3개 정의 완료')
print('  - get_screening_result(): 1차 심사 결과 조회')
print('  - calculate_weighted_score(): 가중치 종합 점수 계산')
print('  - get_shortlist(): 점수 기반 후보 목록 생성')

print('\n도구 단독 테스트:')
print(get_screening_result('20240101'))
print(calculate_weighted_score('20240101'))
print(get_shortlist(min_score=3.5, max_candidates=3))

✅ Selection Recommender 도구 3개 정의 완료
  - get_screening_result(): 1차 심사 결과 조회
  - calculate_weighted_score(): 가중치 종합 점수 계산
  - get_shortlist(): 점수 기반 후보 목록 생성

도구 단독 테스트:
{"student_id": "20240101", "name": "이지은", "essay_score": 4.2, "eligibility": "PASS", "portfolio_score": 3.8}
이지은 (20240101): 종합 점수 = 3.92 (에세이 4.2×0.3 + 포트폴리오 3.8×0.7)
{"shortlist": [{"student_id": "20240404", "name": "최수진", "score": 4.38}, {"student_id": "20240202", "name": "박민준", "score": 4.2}, {"student_id": "20240101", "name": "이지은", "score": 3.92}], "count": 3}


In [ ]:
# ====================================================
# Selection Recommender: ReAct 루프 + 단계별 도구 게이트
# allowed_function_names 로 1->2->3 순서를 결정적으로 강제
# ====================================================
import re

_SELECTION_SYS = '''당신은 Selection Recommender 에이전트입니다.
1차 심사 결과(에세이/자격/포트폴리오)를 탐색하여 면접 후보를 선발하세요.

처리 순서 (시스템이 단계별로 호출 가능한 도구를 제한합니다):
1) 모든 지원자에 대해 get_screening_result 를 한 명씩 호출 (학번 인자 전달)
2) eligibility=PASS 인 지원자에 대해서만 calculate_weighted_score 호출 (FAIL은 부르지 마세요)
3) get_shortlist 를 한 번 호출하여 최종 후보 목록 생성

각 단계에서는 그 단계의 도구만 호출할 수 있습니다. 다음 단계로 넘어가면 자동으로 도구가 바뀝니다.'''


def _make_force_config(allowed: list[str]) -> types.GenerateContentConfig:
    """특정 도구 이름만 호출 가능한 강제 모드 config 생성."""
    return types.GenerateContentConfig(
        tools=[get_screening_result, calculate_weighted_score, get_shortlist],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode='ANY', allowed_function_names=allowed,
            )
        ),
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        thinking_config=types.ThinkingConfig(include_thoughts=True),
        system_instruction=_SELECTION_SYS,
    )


_selection_config_free = types.GenerateContentConfig(
    tools=[get_screening_result, calculate_weighted_score, get_shortlist],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    thinking_config=types.ThinkingConfig(include_thoughts=True),
    system_instruction=_SELECTION_SYS,
)


def run_react_selection(task: str, max_steps: int = 25) -> str:
    """ReAct 루프 + 단계별 도구 게이트로 Selection Recommender를 실행합니다.

    각 step 시작 시 현재 상태(누가 screening 됐는지, 누가 PASS인지, 누가 scoring
    됐는지, shortlist 호출됐는지)를 보고 다음 step에 호출 가능한 도구를
    하나로 제한하여 1->2->3 순서를 결정적으로 강제합니다.

    Args:
        task: 처리할 선발 지시 문자열 (8자리 학번을 자동 추출)
        max_steps: 무한 루프 방지용 최대 반복 횟수

    Returns:
        에이전트 최종 텍스트 응답
    """
    expected_ids = re.findall(r'\d{8}', task)
    print(f'👤 지시: {task[:80]}...')
    print(f'📋 검토 대상 학번: {expected_ids}')
    print('=' * 57)

    messages = [types.Content(role='user', parts=[types.Part(text=task)])]
    screened_ids: set[str] = set()   # screening 호출된 학번
    pass_ids: set[str] = set()       # PASS 판정 학번
    scored_ids: set[str] = set()     # scoring 호출된 학번
    shortlist_done = False

    for step in range(1, max_steps + 1):
        print(f'\n--- Step {step} ---')

        # 현재 상태에 따라 단계 및 허용 도구 결정
        if shortlist_done:
            cfg = _selection_config_free
            print('🔓 단계 4 (자유 모드 — 최종 요약)')
        else:
            unscreened = set(expected_ids) - screened_ids
            unscored = pass_ids - scored_ids
            if unscreened:
                cfg = _make_force_config(['get_screening_result'])
                print(f'🔒 단계 1 — get_screening_result 만 허용 (남은: {len(unscreened)}명)')
            elif unscored:
                cfg = _make_force_config(['calculate_weighted_score'])
                print(f'🔒 단계 2 — calculate_weighted_score 만 허용 (남은: {len(unscored)}명)')
            else:
                cfg = _make_force_config(['get_shortlist'])
                print('🔒 단계 3 — get_shortlist 만 허용')

        response = client.models.generate_content(
            model='gemini-2.5-flash', contents=messages, config=cfg,
        )
        response_content = response.candidates[0].content

        fc_parts = [p for p in response_content.parts
                    if hasattr(p, 'function_call') and p.function_call]

        # function_call 없음 → 최종 답변 (자유 모드에서만 정상)
        if not fc_parts:
            final = response.text
            print(f'\n✅ 최종 선발 결과:\n{final}')
            return final

        for part in response_content.parts:
            if not (hasattr(part, 'text') and part.text):
                continue
            if getattr(part, 'thought', False):
                print(f'💭 Thought: {part.text.strip()}')
            else:
                print(f'💬 Text: {part.text.strip()}')

        fc = fc_parts[0].function_call
        args = dict(fc.args)
        print(f'⚡ Action:  {fc.name}({args})')

        tool_fn = TOOL_REGISTRY_SELECTION.get(fc.name)
        observation = tool_fn(**args) if tool_fn else f'알 수 없는 도구: {fc.name}'
        print(f'👁️  Observation: {observation}')

        # 상태 업데이트 (관찰 결과 파싱)
        raw_sid = args.get('student_id')
        sid = raw_sid.strip() if isinstance(raw_sid, str) else None
        if fc.name == 'get_screening_result' and sid:
            screened_ids.add(sid)
            try:
                obs_json = json.loads(observation)
                if obs_json.get('eligibility') == 'PASS':
                    pass_ids.add(sid)
            except (json.JSONDecodeError, TypeError):
                pass
        elif fc.name == 'calculate_weighted_score' and sid:
            scored_ids.add(sid)
        elif fc.name == 'get_shortlist':
            shortlist_done = True

        messages.append(response_content)
        messages.append(types.Content(
            role='tool',
            parts=[types.Part(function_response=types.FunctionResponse(
                name=fc.name,
                response={'result': observation},
            ))],
        ))

    return '최대 단계 초과'


# ====================================================
# ReAct 테스트
# ====================================================
print('🔄 Selection Recommender (ReAct, 단계별 게이트) 테스트\n')
_ = run_react_selection(
    '학번 20240101, 20240202, 20240303, 20240404, 20240505 지원자들의 '
    '1차 심사 결과를 검토하여 면접 후보 3명을 선발해줘. '
    'eligibility FAIL 지원자는 제외하고, 종합 점수 기준으로 상위 3명을 추천해줘.'
)

## 4. 계획 후 실행 에이전트 (Planner-Executor) — Acceptance Recommender (Task 4)

### APD 적용: 면접 결과 + 1차 심사 → 최종 합격자 결정

복잡한 다단계 결정을 **계획 단계**와 **실행 단계**로 분리합니다.

```
계획 단계 (Planner)
 "어떤 기준으로, 어떤 순서로 최종 합격자를 결정할 것인가?"
 → JSON 계획 수립 (steps + required_tools)

실행 단계 (Executor)
 계획을 전달받아 도구를 사용하여 단계별 실행
 → 최종 합격자 목록 생성
```

### + 쿼리 분해 (Query-Decomposition)

복잡한 판단 질문을 **하위 질문**으로 나눠 순서대로 해결하는 패턴입니다.
"Self-Ask with Search" 패턴이라고도 불립니다.

```
복잡한 질문: "최종 합격자 2명을 어떻게 결정해야 하나?"
 ↓ 분해
 하위 질문 1: "각 지원자의 면접 점수는?"
 하위 질문 2: "1차 심사와 면접 점수를 어떤 비율로 합산해야 하나?"
 하위 질문 3: "동점자 발생 시 어떤 기준을 우선해야 하나?"
 ↓ 종합
 최종 답변: "20240404(최수진), 20240101(이지은) 합격"
```

> **강점**: 명확한 작업 분해, 명시적 계획 기반 디버깅 용이
> **적합**: 긴 작업, 단계별 분해가 필요한 복잡한 결정

In [6]:
# ====================================================
# Acceptance Recommender 도구 정의
# ====================================================

# 가상 면접 결과 DB (Task 3: Human Interview 결과)
INTERVIEW_RESULTS_DB = {
    '20240101': {'name': '이지은', 'interview_score': 4.5,
                 'panel_notes': '논리적 사고 우수, 팀 협업 의지 높음'},
    '20240202': {'name': '박민준', 'interview_score': 4.0,
                 'panel_notes': '기술 지식 탁월, 발표력 양호'},
    '20240404': {'name': '최수진', 'interview_score': 4.8,
                 'panel_notes': '리더십 잠재력, 프로젝트 경험 풍부'},
    '20240505': {'name': '정다은', 'interview_score': 3.7,
                 'panel_notes': '열정적이나 기술 역량 보완 필요'},
}


def get_interview_result(student_id: str) -> str:
    """지원자의 면접 결과를 조회합니다.

    Args:
        student_id: 조회할 지원자 학번

    Returns:
        면접 결과 JSON 문자열. 없으면 오류 메시지.
    """
    result = INTERVIEW_RESULTS_DB.get(student_id.strip())
    if not result:
        return f'학번 {student_id} 면접 결과 없음 (면접 미응시 또는 탈락)'
    return json.dumps({'student_id': student_id, **result}, ensure_ascii=False)


def compute_final_score(student_id: str, doc_weight: float = 0.4,
                        interview_weight: float = 0.6) -> str:
    """1차 심사 점수와 면접 점수를 가중합산하여 최종 점수를 계산합니다.

    Args:
        student_id: 지원자 학번
        doc_weight: 서류 점수 가중치 (기본 0.4)
        interview_weight: 면접 점수 가중치 (기본 0.6)

    Returns:
        최종 점수 계산 결과 문자열
    """
    screening = SCREENING_RESULTS_DB.get(student_id.strip())
    interview = INTERVIEW_RESULTS_DB.get(student_id.strip())

    if not screening:
        return f'학번 {student_id} 서류 심사 결과 없음'
    if screening['eligibility'] == 'FAIL':
        return f'{screening["name"]} ({student_id}): 최종 점수 = 0.00 (자격 미달)'
    if not interview:
        return f'{screening["name"]} ({student_id}): 면접 결과 없음 — 최종 점수 계산 불가'

    doc_score = screening['essay_score'] * 0.3 + screening['portfolio_score'] * 0.7
    final = doc_score * doc_weight + interview['interview_score'] * interview_weight
    return (f'{screening["name"]} ({student_id}): 최종 점수 = {final:.2f} '
            f'(서류 {doc_score:.2f}×{doc_weight} + 면접 {interview["interview_score"]}×{interview_weight})')


def generate_acceptance_report(student_ids_csv: str, headcount: int = 2) -> str:
    """지정 지원자들의 최종 점수를 계산하고 합격자 보고서를 생성합니다.

    Args:
        student_ids_csv: 쉼표로 구분된 학번 목록 (예: '20240101,20240404')
        headcount: 선발 인원 (기본 2명)

    Returns:
        합격자 보고서 JSON 문자열
    """
    # LLM이 float으로 전달할 수 있으므로 int 변환
    headcount = int(headcount)
    id_list = [sid.strip() for sid in student_ids_csv.split(',') if sid.strip()]
    scored = []
    for sid in id_list:
        screening = SCREENING_RESULTS_DB.get(sid, {})
        interview = INTERVIEW_RESULTS_DB.get(sid, {})
        if not screening or screening.get('eligibility') == 'FAIL' or not interview:
            continue
        doc_score = screening['essay_score'] * 0.3 + screening['portfolio_score'] * 0.7
        final = doc_score * 0.4 + interview['interview_score'] * 0.6
        scored.append({'student_id': sid, 'name': screening['name'],
                       'final_score': round(final, 2)})
    scored.sort(key=lambda x: x['final_score'], reverse=True)
    return json.dumps({
        'accepted': scored[:headcount],
        'waitlist': scored[headcount:],
        'headcount': headcount,
    }, ensure_ascii=False)


TOOL_REGISTRY_ACCEPTANCE = {
    'get_interview_result': get_interview_result,
    'compute_final_score': compute_final_score,
    'generate_acceptance_report': generate_acceptance_report,
}

print('✅ Acceptance Recommender 도구 3개 정의 완료')
print('  - get_interview_result(): 면접 결과 조회')
print('  - compute_final_score(): 서류+면접 가중 합산')
print('  - generate_acceptance_report(): 합격자 보고서 생성')

✅ Acceptance Recommender 도구 3개 정의 완료
  - get_interview_result(): 면접 결과 조회
  - compute_final_score(): 서류+면접 가중 합산
  - generate_acceptance_report(): 합격자 보고서 생성


In [ ]:
# ====================================================
# Acceptance Recommender: 계획-실행 패턴 + 쿼리 분해
# ====================================================

_executor_config = types.GenerateContentConfig(
    tools=[get_interview_result, compute_final_score, generate_acceptance_report],
)


def plan_and_accept(task: str, headcount: int = 2) -> str:
    """계획-실행 패턴으로 최종 합격자를 결정합니다.

    Args:
        task: 처리할 합격 결정 지시 문자열
        headcount: 선발 인원

    Returns:
        실행 결과 텍스트
    """
    # Step 1: Planner → JSON 계획 수립
    print('📋 Step 1: 계획 수립 (Planner)...')
    plan_prompt = f'''다음 작업을 처리하기 위한 단계별 계획을 JSON으로 작성하세요.
사용 가능한 도구: get_interview_result, compute_final_score, generate_acceptance_report

작업: {task}

응답 형식 (JSON만):
{{"steps": ["단계1", "단계2", ...], "required_tools": ["도구1", "도구2"], "rationale": "계획 근거"}}'''
    plan_response = client.models.generate_content(model='gemini-2.5-pro', contents=plan_prompt)

    plan_steps_text = task
    try:
        plan_text = plan_response.text.strip()
        if '```json' in plan_text:
            plan_text = plan_text.split('```json')[1].split('```')[0].strip()
        elif '```' in plan_text:
            plan_text = plan_text.split('```')[1].split('```')[0].strip()
        plan = json.loads(plan_text)

        print(f'  계획 수립 완료: {len(plan["steps"])}단계')
        for i, step in enumerate(plan['steps'], 1):
            print(f'  {i}. {step}')

        steps_formatted = '\n'.join(f'  {i}. {s}' for i, s in enumerate(plan['steps'], 1))
        plan_steps_text = (
            f'원래 작업: {task}\n\n'
            f'수립된 실행 계획 (아래 순서를 반드시 따르세요):\n{steps_formatted}\n\n'
            f'필요 도구: {", ".join(plan.get("required_tools", []))}\n\n'
            f'선발 인원: {headcount}명\n\n'
            f'위 계획의 각 단계를 도구를 사용하여 순서대로 실행하세요.'
        )
    except (json.JSONDecodeError, KeyError) as e:
        print(f'  ⚠️ 계획 파싱 실패 ({e}), 원본 작업으로 실행합니다.')

    # Step 2: Executor → 계획대로 도구 실행
    print('\n⚙️  Step 2: 실행 (Executor)...')
    executor_chat = client.chats.create(model='gemini-2.5-flash-lite', config=_executor_config)
    exec_response = executor_chat.send_message(plan_steps_text)

    print('\n✅ 실행 완료')
    return exec_response.text


def query_decomposition(complex_question: str) -> dict:
    """복잡한 합격 결정 질문을 하위 질문으로 분해합니다 (Self-Ask 패턴).

    Args:
        complex_question: 분해할 복잡한 질문

    Returns:
        분해된 하위 질문 목록 딕셔너리
    """
    decomp_prompt = f'''다음 복잡한 질문을 해결하기 위한 하위 질문 목록을 JSON으로 작성하세요.
"자문 후 검색(Self-Ask with Search)" 패턴을 사용하세요:
모델이 스스로 "어떤 후속 질문이 더 필요하지?"를 묻도록 하위 질문을 생성하세요.

복잡한 질문: {complex_question}

응답 형식 (JSON만):
{{"sub_questions": ["질문1", "질문2", "질문3"], "final_synthesis": "최종 답변 도출 방법"}}'''
    response = client.models.generate_content(model='gemini-2.5-flash', contents=decomp_prompt)
    try:
        text = response.text.strip()
        if '```json' in text:
            text = text.split('```json')[1].split('```')[0].strip()
        elif '```' in text:
            text = text.split('```')[1].split('```')[0].strip()
        return json.loads(text)
    except json.JSONDecodeError:
        return {'sub_questions': [], 'final_synthesis': response.text}


# ====================================================
# 테스트 1: Planner-Executor
# ====================================================
print('=' * 57)
print('📋 계획-실행 에이전트 (Planner-Executor) 테스트')
print('=' * 57)
result = plan_and_accept(
    '면접 후보 20240101, 20240202, 20240404, 20240505 중 '
    '최종 합격자 2명을 선발하라. 서류:면접 = 4:6 가중치로 계산하라.',
    headcount=2,
)
print(f'\n📬 최종 결정:\n{result}')

# ====================================================
# 테스트 2: 쿼리 분해
# ====================================================
print('\n' + '=' * 57)
print('🔍 쿼리 분해 (Query-Decomposition) 테스트')
print('=' * 57)
question = '서류 심사와 면접 점수를 어떻게 합산하고, 동점자가 발생할 경우 어떤 기준으로 최종 합격자 2명을 결정해야 하나?'
print(f'복잡한 질문: {question}\n')
decomposed = query_decomposition(question)
print('분해된 하위 질문:')
for i, q in enumerate(decomposed.get('sub_questions', []), 1):
    print(f'  {i}. {q}')
print(f'\n종합 방법: {decomposed.get("final_synthesis", "")[:200]}')

📋 계획-실행 에이전트 (Planner-Executor) 테스트
📋 Step 1: 계획 수립 (Planner)...
  계획 수립 완료: 4단계
  1. 면접 결과 가져오기: 후보자 20240101, 20240202, 20240404, 20240505에 대한 면접 결과를 get_interview_result 도구를 사용하여 가져옵니다.
  2. 점수 계산: 서류 점수와 면접 점수에 4:6 가중치를 적용하여 각 후보자의 최종 점수를 compute_final_score 도구를 사용하여 계산합니다.
  3. 최종 합격자 선발: 계산된 최종 점수를 기준으로 상위 2명의 합격자를 선발합니다.
  4. 합격 보고서 생성: 선발된 최종 합격자에 대한 acceptance_report 도구를 사용하여 합격 보고서를 생성합니다.

⚙️  Step 2: 실행 (Executor)...

✅ 실행 완료

📬 최종 결정:
면접 결과에 따라 계산된 최종 점수는 다음과 같습니다.

*   최수진 (20240404): 최종 점수 4.63 (서류 4.38, 면접 4.8)
*   이지은 (20240101): 최종 점수 4.27 (서류 3.92, 면접 4.5)
*   박민준 (20240202): 최종 점수 4.08 (서류 4.20, 면접 4.0)
*   정다은 (20240505): 최종 점수 3.70 (서류 3.69, 면접 3.7)

서류:면접 = 4:6 가중치로 계산한 결과, 최종 합격자는 최수진 (20240404) 및 이지은 (20240101) 2명입니다.

합격자 보고서:
```json
{"accepted": [{"student_id": "20240404", "name": "최수진", "final_score": 4.63}, {"student_id": "20240101", "name": "이지은", "final_score": 4.27}], "waitlist": [], "headcount": 2}
```

🔍 쿼리 분해 (Query-Decomposition) 테스트
복잡한 질문: 서류 심사와 

## 5. 성찰형 에이전트 (Reflection Agent) — Notification Sender (Task 5)

### APD 적용: 합격/불합격 이메일 품질 자동 보장

최종 합격자 결정 후, Notification Sender가 **자기 평가**를 통해 이메일 품질을 보장합니다.

```
생성 → 자기 평가 → (불합격) → 재생성(피드백 반영) → 자기 평가 → ... → (합격) → 발송
```

**평가 기준 (각 1점씩):**
- ① 결과 명시 (합격/불합격이 첫 문장에 명확한가?)
- ② 사유 제시 (왜 합격/불합격인지 설명했는가?)
- ③ 향후 안내 (다음 단계, 오리엔테이션 일정 등)
- ④ 공감 표현 (지원자 감사 인사, 위로 표현 포함)
- ⑤ 전체 자연스러움

> **강점**: 품질 자동 보장, 오류 자기 복구
> **약점**: LLM 호출 2배 이상 소모 (비용·속도 ↑)
> **적합**: 합/불합격 이메일, 피드백 보고서 등 중요 출력 생성

In [15]:
import re as _re

# ====================================================
# Notification Sender: 성찰형 에이전트
# ====================================================

_notif_gen_config = types.GenerateContentConfig(
    system_instruction='''당신은 동아리 합격/불합격 이메일을 작성하는 Notification Sender입니다.
이메일에는 반드시 아래 4가지를 포함하세요:
① 결과 명시 (합격/불합격을 첫 문장에 명확히)
② 구체적 사유 (왜 이 결과인지 1~2문장)
③ 향후 안내 (합격: 오리엔테이션 일정/준비사항, 불합격: 재지원 안내)
④ 공감 표현 (지원 감사 인사 또는 위로 표현)
이메일은 정중하고 따뜻한 톤으로 작성하세요.''',
)

_notif_eval_config = types.GenerateContentConfig(
    system_instruction='''당신은 동아리 이메일의 품질을 평가하는 전문가입니다.
반드시 아래 JSON 형식으로만 응답하세요:
{"score": 1~5, "passed": true/false, "missing": ["누락 항목"], "reason": "평가 근거", "suggestion": "개선 방향"}

평가 기준 (각 1점씩):
- 결과 명시 (합격/불합격 첫 문장 명시)
- 구체적 사유 (이유 설명)
- 향후 안내 (다음 단계 안내)
- 공감 표현 (감사/위로)
- 전체 자연스러움

passed 기준: score >= 4 AND 결과/사유/향후안내 모두 포함''',
)


def notification_sender_reflective(
    applicant: dict,
    result: str,
    reason: str,
    max_iterations: int = 3,
    pass_threshold: int = 4,
) -> str:
    """성찰형 에이전트: 합격/불합격 이메일을 자기 평가로 품질 보장합니다.

    Args:
        applicant: {'name': '이름', 'email': '이메일주소'}
        result: '합격' 또는 '불합격'
        reason: 합격/불합격 사유
        max_iterations: 최대 재생성 횟수
        pass_threshold: 합격 기준 점수

    Returns:
        품질 기준을 통과한 최종 이메일 텍스트
    """
    context = (
        f'지원자: {applicant["name"]} ({applicant["email"]})\n'
        f'결과: {result}\n'
        f'사유: {reason}'
    )

    print(f'👤 지원자: {applicant["name"]} | 결과: {result}')
    print(f'🎯 합격 기준: {pass_threshold}점 이상 (최대 {max_iterations}회 시도)')
    print('=' * 57)

    last_suggestion = ''
    last_response = ''

    for i in range(1, max_iterations + 1):
        print(f'\n--- 시도 {i}/{max_iterations} ---')

        gen_prompt = f'{context}\n\n위 내용을 바탕으로 이메일을 작성해주세요.'
        if last_suggestion:
            gen_prompt += f'\n\n[이전 응답 개선 피드백]: {last_suggestion}\n피드백을 반영하여 개선하세요.'

        gen_response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=gen_prompt, config=_notif_gen_config)
        last_response = gen_response.text
        preview = last_response[:100] + ('...' if len(last_response) > 100 else '')
        print(f'📝 생성: {preview}')

        eval_prompt = (
            f'지원자: {applicant["name"]}, 결과: {result}\n'
            f'작성된 이메일:\n{last_response}\n\n위 이메일을 평가하세요.'
        )
        eval_response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=eval_prompt, config=_notif_eval_config)

        try:
            eval_text = eval_response.text.strip()
            match = _re.search(r'\{.*\}', eval_text, _re.DOTALL)
            if match:
                eval_text = match.group()
            evaluation = json.loads(eval_text)

            score = evaluation.get('score', 0)
            passed = evaluation.get('passed', False)
            reason_eval = evaluation.get('reason', '')
            missing = evaluation.get('missing', [])
            last_suggestion = evaluation.get('suggestion', '')

            print(f'🔍 평가: {score}/5점 — {reason_eval}')
            if missing:
                print(f'   누락: {missing}')

            if passed or score >= pass_threshold:
                print(f'\n✅ 합격! ({score}점, {i}번 시도)')
                return last_response
            else:
                print(f'❌ 불합격 ({score}점). 개선 방향: {last_suggestion}')

        except (json.JSONDecodeError, KeyError) as e:
            print(f'⚠️ 평가 파싱 오류 ({e}), 현재 응답 반환')
            return last_response

    print('\n⚠️ 최대 시도 횟수 도달. 마지막 응답 반환.')
    return last_response


# ====================================================
# 테스트: 합격 이메일 + 불합격 이메일
# ====================================================
print('🔄 성찰형 에이전트 (Notification Sender) 테스트\n')

result1 = notification_sender_reflective(
    applicant={'name': '최수진', 'email': 'choi@univ.ac.kr'},
    result='합격',
    reason='에세이 기술 이해도 최상위(4.8점), 면접 리더십 잠재력과 프로젝트 경험 탁월',
    max_iterations=3,
)
print(f'\n📬 최종 합격 이메일 (앞 200자):\n{result1[:200]}...')

print('\n' + '─' * 57 + '\n')

result2 = notification_sender_reflective(
    applicant={'name': '정다은', 'email': 'jung@univ.ac.kr'},
    result='불합격',
    reason='기술 역량 보완이 필요하며, 면접 점수(3.7)가 최소 기준(4.0)에 미달',
    max_iterations=3,
)
print(f'\n📬 최종 불합격 이메일 (앞 200자):\n{result2[:200]}...')

🔄 성찰형 에이전트 (Notification Sender) 테스트

👤 지원자: 최수진 | 결과: 합격
🎯 합격 기준: 4점 이상 (최대 3회 시도)

--- 시도 1/3 ---
📝 생성: ## [OO대학교 OO동아리] 지원 결과 안내 (최수진님)

안녕하세요, 최수진님.

[OO대학교 OO동아리]에 지원해주셔서 진심으로 감사드립니다.

기쁜 소식을 전해드리게 되어 ...
🔍 평가: 5/5점 — 이메일이 모든 평가 기준을 충족합니다. 합격/불합격 결과를 첫 문장에 명확하게 제시하였고, 합격 사유를 구체적으로 설명했습니다. 또한, 다음 단계인 오리엔테이션 일정을 상세하게 안내하였으며, 지원자에 대한 감사와 축하의 공감 표현이 자연스럽게 담겨 있습니다. 전체적으로 내용 구성과 표현이 매우 자연스럽고 완성도가 높습니다.

✅ 합격! (5점, 1번 시도)

📬 최종 합격 이메일 (앞 200자):
## [OO대학교 OO동아리] 지원 결과 안내 (최수진님)

안녕하세요, 최수진님.

[OO대학교 OO동아리]에 지원해주셔서 진심으로 감사드립니다.

기쁜 소식을 전해드리게 되어 저희도 매우 설렙니다. **최수진님께서는 OO동아리에 최종 합격하셨음을 알려드립니다.**

지원자님의 에세이에서 보여주신 뛰어난 기술 이해도(4.8점)와 면접에서 보여주신 리더십 ...

─────────────────────────────────────────────────────────

👤 지원자: 정다은 | 결과: 불합격
🎯 합격 기준: 4점 이상 (최대 3회 시도)

--- 시도 1/3 ---
📝 생성: ## 동아리 지원 결과 안내 (정다은님)

안녕하세요, 정다은님.

동아리 [동아리 이름] 지원해주셔서 진심으로 감사드립니다.

안타깝게도 이번 [동아리 이름] 신입 부원 모집 결...
🔍 평가: 5/5점 — 모든 평가 기준을 충족하며, 명확하고 정중한 태도로 이메일을 작성했습니다. 결과 명시, 구체적 사유, 향후 안내, 공감 표현, 전체적인 자연스러움 모두 훌륭합니다.

✅ 합격! (5점,

## 6. 에이전트 유형 비교 요약

| 유형 | APD 에이전트 | 핵심 특징 | 비용 | 적합한 상황 |
|------|------------|---------|------|------------|
| **반사형** | Application Router | LLM 없음, 규칙만 | 거의 없음 | 빠른 라우팅, 형식 검사 |
| **ReAct** | Selection Recommender | Thought→Action→Obs 루프 | 중간 | 다단계 탐색, 동적 분석 |
| **계획형** | Acceptance Recommender | 계획 JSON → 실행 분리 | 중간 | 복잡한 다단계 결정 |
| **쿼리 분해** | Acceptance Recommender (확장) | 복잡한 질문 → 하위 질문 | 중간 | 외부 지식 검색 필요한 분석 |
| **성찰형** | Notification Sender | 생성→평가→재생성 | 높음 | 품질이 중요한 문서 생성 |

### 에이전트 유형 선택 가이드

```python
if 응답_속도가_최우선 and 규칙이_명확:
 → 반사형 (Application Router)
elif 다단계_추론 and 탐색적_시나리오:
 → ReAct (Selection Recommender)
elif 복잡한_다단계_작업 and 계획_수립_필요:
 → 계획형 (Acceptance Recommender)
elif 품질보장 and 재시도_허용:
 → 성찰형 (Notification Sender)
```

## 7. 멀티 에이전트: APD 동아리 모집 파이프라인 전체 실행

지금까지 구현한 모든 에이전트를 연결하여 APD 파이프라인을 실행합니다.

```
지원서 입력
 │
 ▼
[반사형] Application Router
 │ STANDARD (오류/해외 시 즉각 반려)
 ▼
[exercise 2] Application Collector → 유효성 검사 + 정규화
 │
 ├─ [exercise 1] Eligibility Checker (Task 1b)
 ├─ [exercise 2] Essay Analyzer (Task 1a) ← 병렬 실행 (섹션 8)
 └─ [exercise 2] Portfolio Evaluator (Task 1c)
 │
 ▼
[ReAct] Selection Recommender → 면접 후보 선발
 │
 ▼ (면접 실시 — Human 수행, 결과는 DB에 입력)
 │
 ▼
[계획형] Acceptance Recommender → 최종 합격자 결정
 │
 ▼
[성찰형] Notification Sender → 개인화 이메일 발송
 │
 ▼
OUTPUT (합격자 확정)
```

> **핵심**: `BaseAgentWorker.run()` 공통 인터페이스 덕분에
> 오케스트레이터가 에이전트 유형에 관계없이 동일하게 호출합니다.

In [17]:
# ====================================================
# 순차 파이프라인 오케스트레이터
# ※ Task 1 병렬 처리는 다음 섹션에서 다룹니다.
# ====================================================

def run_recruitment_pipeline_sequential(application: dict) -> dict:
    """APD 동아리 모집 파이프라인을 순차 실행합니다.

    Args:
        application: {'name': ..., 'student_id': ..., 'major': ..., 'courses': ...,
                      'year': ..., 'essay': ..., 'portfolio_url': ..., 'email': ...}

    Returns:
        각 단계 처리 결과 딕셔너리
    """
    results = {'applicant': application['name'], 'student_id': application['student_id']}
    print(f'\n🚀 파이프라인 시작: {application["name"]} ({application["student_id"]})')
    print('=' * 57)

    # Step 0: 반사형 라우터
    print('\n[Step 0] 반사형 라우터 실행...')
    route_result = reflex_application_router(
        application['student_id'], application['major'], application.get('courses', '')
    )
    results['routing'] = route_result
    print(f'  결과: {route_result["message"]}')

    if route_result['category'] in ('ERROR', 'INCOMPLETE'):
        results['status'] = 'REJECTED_AT_ROUTING'
        print('  → 파이프라인 중단 (즉각 반려)')
        return results

    # Step 1: Application Collector (exercise 2)
    print('\n[Step 1] Application Collector 실행...')
    collector = ApplicationCollectorAgent()
    msg = (
        f'이름={application["name"]}, 학번={application["student_id"]}, '
        f'전공={application["major"]}, 이수과목={application.get("courses", "")}를 처리해줘'
    )
    collection_result = collector.run(msg)
    results['collection'] = collection_result[:150]
    print(f'  결과 (앞 100자): {collection_result[:100]}...')

    # Step 1b: Eligibility Checker (exercise 1)
    print('\n[Step 1b] Eligibility Checker 실행...')
    checker = EligibilityCheckerAgent()
    eligibility_result = checker.run(
        f'{application["name"]}, {application.get("year", "?")}, '
        f'{application["major"]}, {application.get("courses", "없음")}'
    )
    results['eligibility'] = eligibility_result[:150]
    print(f'  결과 (앞 100자): {eligibility_result[:100]}...')

    if 'FAIL' in eligibility_result.upper():
        # 불합격 이메일 발송
        print('\n  → 자격 미달. 불합격 이메일 발송 중...')
        email = notification_sender_reflective(
            applicant={'name': application['name'],
                       'email': application.get('email', 'applicant@univ.ac.kr')},
            result='불합격',
            reason='자격 기준 미충족 (학년/전공/이수과목 중 1개 이상 미달)',
            max_iterations=2,
        )
        results['notification'] = email[:200]
        results['status'] = 'REJECTED_AT_ELIGIBILITY'
        return results

    print('\n[Step 2] Selection Recommender (ReAct) — 사전 구축된 DB 기준으로 진행')
    print('  (실제: 위 심사 결과가 SCREENING_RESULTS_DB에 저장된 후 실행)')
    results['shortlist'] = 'Selection Recommender 처리 완료'

    print('\n[Step 4] Acceptance Recommender (계획형) — 사전 구축된 DB 기준으로 진행')
    results['acceptance'] = 'Acceptance Recommender 처리 완료'

    print('\n[Step 5] Notification Sender (성찰형) 실행...')
    email_result = notification_sender_reflective(
        applicant={'name': application['name'],
                   'email': application.get('email', 'applicant@univ.ac.kr')},
        result='합격',
        reason='에세이 및 포트폴리오 심사 통과, 자격 기준 충족',
        max_iterations=2,
    )
    results['notification'] = email_result[:200]
    results['status'] = 'COMPLETED'

    print('\n' + '=' * 57)
    print(f'✅ 파이프라인 완료: {application["name"]}')
    return results


# ====================================================
# 순차 파이프라인 테스트
# ====================================================
print('🕸️ APD 동아리 모집 파이프라인 (순차) 테스트')

test_applicant = {
    'name': '이지은',
    'student_id': '20249999',
    'major': '컴퓨터공학과',
    'year': '2학년',
    'courses': '파이썬 기초, 선형대수',
    'essay': 'AI와 머신러닝에 관심이 많습니다.',
    'portfolio_url': 'https://scaler-blog-prod-wp-content.s3.ap-south-1.amazonaws.com/wp-content/uploads/2024/08/22163741/Maggie-Wolff-1024x501.png',
    'email': 'lee@univ.ac.kr',
}
pipeline_result = run_recruitment_pipeline_sequential(test_applicant)
print(f'\n📋 최종 처리 상태: {pipeline_result["status"]}')

🕸️ APD 동아리 모집 파이프라인 (순차) 테스트

🚀 파이프라인 시작: 이지은 (20249999)

[Step 0] 반사형 라우터 실행...
  결과: [반사형] ✅ 정상 접수 → Application Collector로 전달

[Step 1] Application Collector 실행...
  결과 (앞 100자): 제출해주신 지원서가 정상적으로 접수되었습니다.
성명: 이지은
학번: 20249999
전공: 컴퓨터공학과
이수과목: 파이썬 기초, 선형대수...

[Step 1b] Eligibility Checker 실행...
  결과 (앞 100자): - 학년 기준: ✅ 합격
- 전공 기준: ✅ 합격
- 이수 과목: ✅ 합격
- 최종 판정: **PASS**...

[Step 2] Selection Recommender (ReAct) — 사전 구축된 DB 기준으로 진행
  (실제: 위 심사 결과가 SCREENING_RESULTS_DB에 저장된 후 실행)

[Step 4] Acceptance Recommender (계획형) — 사전 구축된 DB 기준으로 진행

[Step 5] Notification Sender (성찰형) 실행...
👤 지원자: 이지은 | 결과: 합격
🎯 합격 기준: 4점 이상 (최대 2회 시도)

--- 시도 1/2 ---
📝 생성: ## 동아리 [동아리 이름] 합격 안내

**[지원자 이름]님께,**

안녕하십니까. [동아리 이름] 동아리입니다.

먼저 저희 동아리에 관심을 가지고 지원해 주셔서 진심으로 감사...
🔍 평가: 5/5점 — 결과 명시, 구체적 사유, 향후 안내, 공감 표현, 전체 자연스러움 모든 기준을 충족하여 높은 점수를 부여합니다.

✅ 합격! (5점, 1번 시도)

✅ 파이프라인 완료: 이지은

📋 최종 처리 상태: COMPLETED


## 8. 병렬 패턴: Task 1a / 1b / 1c 동시 실행

APD 파이프라인에서 Task 1a / 1b / 1c는 **독립적**입니다.
`ThreadPoolExecutor`로 세 에이전트를 동시에 실행하면 전체 처리 시간을 단축할 수 있습니다.

```
순차 실행: [Essay Analyzer] → [Eligibility Checker] → [Portfolio Evaluator]
 총 시간 = T_essay + T_eligibility + T_portfolio (≈ 3T)

병렬 실행: [Essay Analyzer] ─┐
 [Eligibility Checker] ─┼→ [결과 통합]
 [Portfolio Evaluator] ─┘
 총 시간 ≈ max(T_essay, T_eligibility, T_portfolio) (≈ 1T)
```

> `google-genai` SDK의 `generate_content()`는 **동기** 함수입니다.
> `ThreadPoolExecutor`로 여러 동기 함수를 스레드에서 동시에 **비동기** 실행합니다.

In [21]:
# ====================================================
# 병렬 심사: ThreadPoolExecutor로 Task 1a / 1b / 1c 동시 실행
# ====================================================
import concurrent.futures

def task1a_essay(essay_text: str) -> dict:
    print('  ▶ [Task 1a: Essay Analyzer] 시작')
    try:
        agent = EssayAnalyzerAgent()
        result = agent.run(essay_text)
    except Exception as e:
        result = f'[Task 1a 오류: {type(e).__name__}] {str(e)[:100]}'
    print('  ◀ [Task 1a: Essay Analyzer] 완료')
    return {'task': '1a', 'agent': 'Essay Analyzer', 'result': result[:200]}


def task1b_eligibility(applicant_info: str) -> dict:
    print('  ▶ [Task 1b: Eligibility Checker] 시작')
    try:
        agent = EligibilityCheckerAgent()
        result = agent.run(applicant_info)
    except Exception as e:
        result = f'[Task 1b 오류: {type(e).__name__}] {str(e)[:100]}'
    print('  ◀ [Task 1b: Eligibility Checker] 완료')
    return {'task': '1b', 'agent': 'Eligibility Checker', 'result': result[:200]}


def task1c_portfolio(image_url: str) -> dict:
    print('  ▶ [Task 1c: Portfolio Evaluator] 시작')
    try:
        agent = PortfolioEvaluatorAgent()
        result = agent.run(image_url)
    except Exception as e:
        result = f'[Task 1c 오류: {type(e).__name__}] {str(e)[:100]}'
    print('  ◀ [Task 1c: Portfolio Evaluator] 완료')
    return {'task': '1c', 'agent': 'Portfolio Evaluator', 'result': result[:200]}


def parallel_screening(applicant: dict) -> dict:
    print(f'👤 지원자: {applicant["name"]} | 3개 에이전트 병렬 실행')
    print('=' * 57)

    t_start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        future_essay = executor.submit(task1a_essay, applicant.get('essay', ''))
        future_elig = executor.submit(
            task1b_eligibility,
            f'{applicant["name"]}, {applicant.get("year", "?")}, '
            f'{applicant["major"]}, {applicant.get("courses", "없음")}'
        )
        future_port = executor.submit(task1c_portfolio, applicant.get('portfolio_url', ''))

        def safe_result(future, idx):
            try:
                return future.result()
            except Exception as e:
                return {'task': f'1{chr(97+idx)}', 'agent': '?', 'result': f'[오류: {type(e).__name__}]'}

        essay_res = safe_result(future_essay, 0)
        eligibility_res = safe_result(future_elig, 1)
        portfolio_res = safe_result(future_port, 2)

    t_elapsed = time.time() - t_start
    print(f'\n⏱️  병렬 완료: {t_elapsed:.1f}초')

    print('\n🔗 결과 통합 중...')
    _combiner_config = types.GenerateContentConfig(
        system_instruction='세 심사 에이전트의 결과를 한 문단으로 통합 요약하세요.',
    )
    combine_prompt = (
        f'지원자 {applicant["name"]}의 1차 심사 결과 통합:\n\n'
        f'[Task 1a 에세이 심사]\n{essay_res["result"]}\n\n'
        f'[Task 1b 자격 검사]\n{eligibility_res["result"]}\n\n'
        f'[Task 1c 포트폴리오 평가]\n{portfolio_res["result"]}\n\n'
        '세 심사 결과를 통합하여 종합 의견을 2~3문장으로 작성하세요.'
    )
    integrated = client.models.generate_content(
        model='gemini-2.5-flash-lite', config=_combiner_config,
        contents=combine_prompt
    )
    print(f'\n✅ 통합 의견 (앞 200자):\n{integrated.text[:200]}...')

    return {
        'applicant': applicant['name'],
        'task1a': essay_res,
        'task1b': eligibility_res,
        'task1c': portfolio_res,
        'integrated': integrated.text,
        'elapsed_sec': round(t_elapsed, 1),
    }


# ====================================================
# 병렬 실행 테스트
# ====================================================
print('🔄 병렬 심사 패턴 (ThreadPoolExecutor) 테스트\n')

test_parallel = {
    'name': '박민준',
    'student_id': '20240202',
    'major': '물리학과',
    'year': '3학년',
    'courses': '선형대수',
    'essay': '저는 강화학습과 대규모 언어 모델에 관심이 많습니다. 멀티에이전트 시스템을 연구하고 싶어 지원했습니다.',
    'portfolio_url': 'https://scaler-blog-prod-wp-content.s3.ap-south-1.amazonaws.com/wp-content/uploads/2024/08/22163741/Maggie-Wolff-1024x501.png',
}

parallel_result = parallel_screening(test_parallel)
print(f'\n처리 시간: {parallel_result["elapsed_sec"]}초')

🔄 병렬 심사 패턴 (ThreadPoolExecutor) 테스트

👤 지원자: 박민준 | 3개 에이전트 병렬 실행
  ▶ [Task 1a: Essay Analyzer] 시작
  ▶ [Task 1b: Eligibility Checker] 시작
  ▶ [Task 1c: Portfolio Evaluator] 시작
  ◀ [Task 1b: Eligibility Checker] 완료
  ◀ [Task 1a: Essay Analyzer] 완료
  ◀ [Task 1c: Portfolio Evaluator] 완료

⏱️  병렬 완료: 16.3초

🔗 결과 통합 중...

✅ 통합 의견 (앞 200자):
지원자 박민준은 1차 심사에서 전반적으로 우수한 평가를 받았습니다. Task 1a 에세이 심사에서는 강화학습, 대규모 언어 모델, 멀티에이전트 시스템 등 AI 핵심 기술에 대한 이해도가 높았으며, Task 1b 자격 검사에서도 학년, 전공, 이수 과목 모두 기준을 충족하여 합격했습니다. 또한 Task 1c 포트폴리오 평가는 기술 수준 5점, AI 관련성 4...

처리 시간: 16.3초


## 9. 실습 과제

### 과제 1: Selection Recommender 가중치 실험
`calculate_weighted_score()`에서 `essay_weight`와 `portfolio_weight`를 변경하면
후보 순위가 어떻게 달라지는지 확인하세요.

### 과제 2: Notification Sender 평가 기준 강화
`notification_evaluator`의 시스템 프롬프트에 **"오탈자 없음"** 기준을 추가하고
`pass_threshold`를 5점으로 올려보세요.

### 과제 3: 파이프라인 오류 처리 강화 (선택)
`run_recruitment_pipeline_sequential()`에서 학번 형식 오류 지원자에게도
안내 이메일을 발송하도록 수정하세요.

### 과제 4: Interview Scheduler 구현 (선택)
APD Task 3 에이전트인 **Interview Scheduler**를 `BaseAgentWorker`를 상속하여 구현하세요.
(힌트: `find_available_slots()`, `send_interview_invitation()` 가상 도구를 정의하세요)

In [20]:
# ====================================================
# 과제 1: 가중치 실험
# ====================================================
# TODO: 아래 값을 변경하고 순위 변화를 관찰하세요
MY_ESSAY_WEIGHT = 0.3       # 예: 0.5로 바꿔보기
MY_PORTFOLIO_WEIGHT = 0.7   # 예: 0.5로 바꿔보기

print(f'가중치: 에세이 {MY_ESSAY_WEIGHT} | 포트폴리오 {MY_PORTFOLIO_WEIGHT}')
for sid, data in sorted(SCREENING_RESULTS_DB.items()):
    if data['eligibility'] == 'FAIL':
        print(f'  {data["name"]} ({sid}): SKIP (자격 미달)')
        continue
    score = data['essay_score'] * MY_ESSAY_WEIGHT + data['portfolio_score'] * MY_PORTFOLIO_WEIGHT
    print(f'  {data["name"]} ({sid}): {score:.2f}점')


# ====================================================
# 과제 2: 평가 기준 강화
# ====================================================
# TODO: 아래 프롬프트에 "오탈자 없음" 기준을 추가하고 pass_threshold를 5로 올려보세요
MY_EVALUATOR_PROMPT = '''당신은 동아리 이메일의 품질을 평가하는 전문가입니다.
반드시 아래 JSON 형식으로만 응답하세요:
{"score": 1~5, "passed": true/false, "missing": ["누락 항목"], "reason": "평가 근거", "suggestion": "개선 방향"}

평가 기준 (각 1점씩):
- 결과 명시: 0 또는 1점
- 구체적 사유: 0 또는 1점
- 향후 안내: 0 또는 1점
- 공감 표현: 0 또는 1점
- 전체 자연스러움: 0 또는 1점
# TODO: 6번째 기준 추가 → 오탈자 없음 (문법/맞춤법): 0 또는 1점

passed 기준: score >= 4  # TODO: 5로 변경해보기
'''

my_evaluator_config = types.GenerateContentConfig(system_instruction=MY_EVALUATOR_PROMPT)
print('강화된 평가 기준 모델 생성 완료')


# ====================================================
# 과제 4: Interview Scheduler (선택)
# ====================================================

def find_available_slots(candidate_id: str, week: str) -> str:
    '''지원자와 면접관의 공통 가용 시간대를 조회합니다.
    Args:
        candidate_id: 지원자 학번
        week: 면접 예정 주 (예: "2024-05-01")
    Returns:
        가용 시간대 목록 문자열
    '''
    # TODO: 가상 Calendar DB에서 공통 가용 시간대 반환
    pass

def send_interview_invitation(candidate_id: str, slot: str) -> str:
    '''지원자에게 면접 초대를 발송합니다.
    Args:
        candidate_id: 지원자 학번
        slot: 면접 시간 (예: "2024-05-08 14:00")
    Returns:
        발송 결과 문자열
    '''
    # TODO: 가상 이메일/알림 발송 구현
    pass

class InterviewSchedulerAgent(BaseAgentWorker):
    '''Task 3 — Interview Scheduler.'''
    def __init__(self):
        # TODO: super().__init__() 호출 및 모델 설정
        pass
    def run(self, candidate_info: str) -> str:
        # TODO: find_available_slots + send_interview_invitation 도구로 스케줄링
        pass

print('과제 4 템플릿 로드 완료. TODO를 완성하세요.')

가중치: 에세이 0.3 | 포트폴리오 0.7
  이지은 (20240101): 3.92점
  박민준 (20240202): 4.20점
  김철수 (20240303): SKIP (자격 미달)
  최수진 (20240404): 4.38점
  정다은 (20240505): 3.69점
강화된 평가 기준 모델 생성 완료
과제 4 템플릿 로드 완료. TODO를 완성하세요.


## 10. 핵심 정리

| 개념 | 핵심 내용 | APD 에이전트 |
|------|----------|------------|
| **반사형** | LLM 없이 규칙만. 밀리초 응답, 예측 가능 | Application Router |
| **ReAct** | Thought→Action→Obs 루프 수동 제어 | Selection Recommender |
| **계획형** | Planner(JSON 계획) → Executor(실행) 분리 | Acceptance Recommender |
| **쿼리 분해** | 복잡한 질문을 하위 질문으로 분해 후 종합 | Acceptance Recommender (확장) |
| **성찰형** | 생성→평가→재생성 루프, 품질 자동 보장 | Notification Sender |
| **BaseAgentWorker** | `run()` 공통 인터페이스 → polymorphic 호출 | 모든 에이전트 |
| **멀티 에이전트** | 역할 분리 + 순차/병렬 조합 | APD 전체 파이프라인 |
| **병렬 패턴** | `ThreadPoolExecutor`로 독립 에이전트 동시 실행 | Task 1a/1b/1c |

### 에이전트 유형 선택 기준

| 상황 | 추천 유형 |
|------|----------|
| 빠른 라우팅, 규칙이 명확 | 반사형 |
| 다단계 탐색, 여러 도구 순차 사용 | ReAct |
| 긴 작업, 단계별 분해 필요 | 계획형 |
| 복잡한 질문을 쪼개서 해결 | 쿼리 분해 |
| 품질이 중요한 문서/이메일 생성 | 성찰형 |
| 독립적 작업을 동시에 처리 | 병렬 패턴 |

---
**다음 exercise 예고**: 에이전트 평가 & 시스템 개선 — 얼마나 잘 작동하는지 측정하기